In [ ]:
# 1. Установка PyTorch и базовых библиотек
!pip install torch==2.2.0 torchvision==0.17.0 --index-url https://download.pytorch.org/whl/cu121
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.2.0+cu121.html

# Принудительно ставим правильный NumPy
!pip install "numpy<2" --force-reinstall
!pip install Pillow>=10.2.0

# 2. Клонирование ODIN и чистка конфликтов
!git clone https://github.com/ayushjain1144/odin.git
%cd /kaggle/working/odin
!sed -i 's/pyyaml==5.3.1/pyyaml>=5.4.1/gi' requirements.txt
!sed -i '/detectron2/d' requirements.txt
!sed -i '/pytorch3d/d' requirements.txt
!pip install -r requirements.txt
!pip install ninja fvcore iopath

# 3. Сборка тяжелых зависимостей (Detectron2 + PyTorch3D)
!pip install git+https://github.com/facebookresearch/detectron2.git
!FORCE_CUDA=1 pip install git+https://github.com/facebookresearch/pytorch3d.git

# 4. === КРИТИЧЕСКИ ВАЖНО: СБОРКА CUDA-ЯДЕР ПОД ТВОЙ GPU ===
%cd /kaggle/working/odin/libs/pointops2
!rm -rf build dist *.egg-info
!TORCH_CUDA_ARCH_LIST="6.0;7.0;7.5;8.0;8.6" python setup.py install --user
%cd /kaggle/working/odin

In [ ]:
import numpy as np
import os

# Защита: проверяем, что рестарт прошел успешно и NumPy правильный
assert np.__version__.startswith('1.'), "NumPy всё ещё версии 2.x! Сделайте Restart Session."

# Скачиваем веса
os.makedirs("/kaggle/working/odin/weights", exist_ok=True)
!wget -nc -q https://huggingface.co/katefgroup/odin/resolve/main/scannet_swin_semantic_77.8_64k_2k.pth -O /kaggle/working/odin/weights/scannet_swin.pth
!wget -nc -q https://huggingface.co/katefgroup/odin/resolve/main/m2f_coco_swin.pkl -O /kaggle/working/odin/weights/m2f_coco_swin.pkl

print("✅ Окружение готово. Веса загружены.")

In [ ]:
import os
import sys
from detectron2.data import MetadataCatalog, DatasetCatalog

REPO_PATH = '/kaggle/working/odin'
os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

SCANNET_CLASSES = [
    "wall", "floor", "cabinet", "bed", "chair", "sofa", "table", "door", 
    "window", "bookshelf", "picture", "counter", "desk", "curtain", 
    "refrigerator", "showercurtain", "toilet", "sink", "bathtub", "otherfurniture"
]

DATASET_NAME = "odin_inference_ds"

# Безопасная очистка и регистрация
if DATASET_NAME in DatasetCatalog.list():
    DatasetCatalog.remove(DATASET_NAME)
try: 
    MetadataCatalog.remove(DATASET_NAME)
except: 
    pass
    
DatasetCatalog.register(DATASET_NAME, lambda: [])
MetadataCatalog.get(DATASET_NAME).set(thing_classes=SCANNET_CLASSES)

print(f"✅ Датасет '{DATASET_NAME}' зарегистрирован.")

In [ ]:
import torch
import cv2
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.modeling import build_model
from detectron2.structures import Instances
from detectron2.projects.deeplab import add_deeplab_config
from odin import add_maskformer2_config, add_maskformer2_video_config

class ODINPredictor:
    def __init__(self, config_path, weights_path):
        cfg = get_cfg()
        add_deeplab_config(cfg)
        add_maskformer2_config(cfg)
        add_maskformer2_video_config(cfg)
        cfg.merge_from_file(config_path)
        
        # Строгая настройка архитектуры
        cfg.DATASETS.TRAIN = (DATASET_NAME,)
        cfg.MODEL.WEIGHTS = weights_path
        cfg.MODEL.DECODER_3D = True
        cfg.INPUT.VOXELIZE = True 
        cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES = 20
        cfg.USE_GHOST_POINTS = False
        cfg.USE_SEGMENTS = False
        cfg.MODEL.MASK_FORMER.TEST.SEMANTIC_ON = True
        cfg.MODEL.MASK_FORMER.TEST.INSTANCE_ON = True
        
        self.model = build_model(cfg)
        self.model.eval()
        DetectionCheckpointer(self.model).load(weights_path)
        
        self.cfg = cfg
        self.device = self.model.device

    def _get_xyz_at_scale(self, depth, intr, pose, scale):
        h, w = depth.shape
        y, x = torch.meshgrid(torch.arange(h, device=self.device), 
                              torch.arange(w, device=self.device), indexing='ij')
        
        fx, fy = intr[0, 0] / scale, intr[1, 1] / scale
        cx, cy = intr[0, 2] / scale, intr[1, 2] / scale
        
        X = (x.float() - cx) * depth / fx
        Y = (y.float() - cy) * depth / fy
        
        pts = torch.stack([X, Y, depth], dim=-1).view(-1, 3)
        pts_homo = torch.cat([pts, torch.ones_like(pts[:, :1])], dim=-1)
        world_pts = (pose.to(self.device) @ pts_homo.T).T[:, :3]
        return world_pts.view(h, w, 3)

    @torch.no_grad()
    def predict(self, frames_list):
        h_orig, w_orig = 480, 640
        
        # Настраиваем модель ровно на 4 уровня вокселей для ResNet пирамиды
        ratios = [32, 16, 8, 4] 
        self.cfg.INPUT.VOXEL_SIZE = [0.32, 0.16, 0.08, 0.04] 

        batch = {
            "file_name": "inference/scene_00/frame_00.jpg", 
            "dataset_name": DATASET_NAME,
            "images": [], "depths": [], "poses": [], "intrinsics": [],
            "decoder_3d": True, "all_classes": SCANNET_CLASSES, "num_classes": 20,
            "instances_all": [], "multiplier": 1000,
            "multi_scale_xyz": [] 
        }

        # Обработка кадров
        for f in frames_list:
            img = cv2.resize(f['image'], (w_orig, h_orig))
            batch["images"].append(torch.as_tensor(img.transpose(2,0,1)).float().to(self.device))
            
            depth = cv2.resize(f['depth'], (w_orig, h_orig), interpolation=cv2.INTER_NEAREST)
            depth_m = torch.as_tensor(depth.astype(np.float32)).to(self.device) / 1000.0
            batch["depths"].append(depth_m)
            
            batch["poses"].append(torch.as_tensor(f['pose']).float().to(self.device))
            batch["intrinsics"].append(torch.as_tensor(f['intrinsic']).float().to(self.device))
            
            # Заглушка для инстансов
            inst = Instances((h_orig, w_orig))
            inst.gt_classes = torch.tensor([], dtype=torch.long, device=self.device)
            inst.instance_ids = torch.tensor([], dtype=torch.long, device=self.device)
            batch["instances_all"].append(inst)

        # Расчет 3D-масштабов
        for r in ratios:
            scale_frames = []
            h_s, w_s = h_orig // r, w_orig // r
            for i in range(len(frames_list)):
                d_s = torch.nn.functional.interpolate(
                    batch["depths"][i][None, None], size=(h_s, w_s), mode='nearest'
                ).squeeze()
                xyz_s = self._get_xyz_at_scale(d_s, batch["intrinsics"][i], batch["poses"][i], r)
                scale_frames.append(xyz_s)
            batch["multi_scale_xyz"].append(torch.stack(scale_frames))

        with torch.cuda.amp.autocast():
            output = self.model([batch])[0]
        
        return output, batch["depths"], batch["poses"]

In [ ]:
import plotly.graph_objects as go

# 1. Инициализируем модель
print("Загрузка архитектуры и весов...")
predictor = ODINPredictor(
    config_path="configs/scannet_context/swin_3d.yaml",
    weights_path="weights/scannet_swin.pth"
)

# 2. Грузим данные
SCENE_PATH = "/kaggle/input/datasets/tiantiansyrinx1102/scannet-data/scannet/posed_images/scene0000_00"
frames = []
for i in [0, 10, 20, 30, 40]:
    f_id = f"{i:05d}"
    frames.append({
        'image': cv2.imread(f"{SCENE_PATH}/{f_id}.jpg"),
        'depth': cv2.imread(f"{SCENE_PATH}/{f_id}.png", cv2.IMREAD_ANYDEPTH),
        'pose': np.loadtxt(f"{SCENE_PATH}/{f_id}.txt"),
        'intrinsic': np.array([[577.8, 0, 319.5, 0], [0, 577.8, 239.5, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
    })

# 3. ИНФЕРЕНС
print("Запускаем 3D-предсказание (это задействует скомпилированные CUDA-ядра)...")
output, depths, poses = predictor.predict(frames)

print(f"✅ Успех! Найдено 3D сегментов: {len(output['instances_3d']['pred_classes'])}")

# 4. Визуализация облака точек первого кадра с цветами сегментации
h, w = 480, 640
y, x = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
depth0 = depths[0].cpu().numpy()
x_3d = (x - 319.5) * depth0 / 577.8
y_3d = (y - 239.5) * depth0 / 577.8
pts = np.stack([x_3d, y_3d, depth0], axis=-1).reshape(-1, 3)

# Берем предсказанные классы (семантика)
colors = output['semantic_3d'].cpu().numpy()

# Фильтруем нулевую глубину, чтобы график был чистым
valid_mask = pts[:, 2] > 0
pts = pts[valid_mask]
colors = colors[valid_mask]

# Отрисовываем каждый 5-й пиксель для скорости
fig = go.Figure(data=[go.Scatter3d(
    x=pts[::5, 0], y=pts[::5, 1], z=pts[::5, 2],
    mode='markers',
    marker=dict(size=2, color=colors[::5], colorscale='Jet')
)])
fig.update_layout(title="ODIN Semantic 3D Reconstruction", scene=dict(aspectmode='data'))
fig.show()

In [ ]:
import os
import torch
import numpy as np
import cv2
from detectron2.data import DatasetCatalog, MetadataCatalog

def load_scannet_flat_5_frames(scene_dir, frame_indices=[0, 10, 20, 30, 40]):
    """
    Загружает 5 кадров из плоской структуры датасета posed_images (ScanNet).
    """
    scene_dict = {
        "file_name": os.path.basename(scene_dir), 
        "dataset_name": "scannet_real_sample",
        "images": [],
        "depths": [],
        "poses": [],
        "intrinsics": [],
        "decoder_3d": True # Важный флаг для ODIN
    }
    
    # Стандартная матрица интринсиков для ScanNet
    default_intrinsic = np.array([
        [577.8706, 0.0,      319.5, 0.0],
        [0.0,      577.8706, 239.5, 0.0],
        [0.0,      0.0,      1.0,   0.0],
        [0.0,      0.0,      0.0,   1.0]
    ], dtype=np.float32)
    intrinsic_tensor = torch.from_numpy(default_intrinsic)

    for idx in frame_indices:
        # Форматируем индекс как строку из 5 цифр (например, 10 -> "00010")
        frame_str = f"{idx:05d}"
        
        # 1. Читаем RGB (00000.jpg)
        color_path = os.path.join(scene_dir, f"{frame_str}.jpg")
        img = cv2.imread(color_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float()
        scene_dict["images"].append(img_tensor)
        
        # 2. Читаем Глубину (00000.png)
        depth_path = os.path.join(scene_dir, f"{frame_str}.png")
        depth_img = cv2.imread(depth_path, cv2.IMREAD_ANYDEPTH) 
        # Конвертируем глубину из миллиметров в метры
        depth_tensor = torch.from_numpy(depth_img.astype(np.float32)) / 1000.0
        scene_dict["depths"].append(depth_tensor)
        
        # 3. Читаем Позу (00000.txt)
        pose_path = os.path.join(scene_dir, f"{frame_str}.txt")
        with open(pose_path, 'r') as f:
            pose_matrix = np.loadtxt(f)
        
        # Защита от "битых" кадров (иногда pose не смогла просчитаться и равна inf)
        if np.isinf(pose_matrix).any():
            pose_matrix = np.eye(4)
            
        pose_tensor = torch.from_numpy(pose_matrix).float()
        scene_dict["poses"].append(pose_tensor)
        
        # 4. Добавляем Интринсики
        scene_dict["intrinsics"].append(intrinsic_tensor)

    return [scene_dict]

# ==== Регистрация датасета ====
# Укажи точный путь к папке scene0000_00 в твоем окружении Kaggle
SCENE_PATH = "/kaggle/input/scannet-data/scannet/posed_images/scene0000_00"

if "scannet_real_sample" in DatasetCatalog.list():
    DatasetCatalog.remove("scannet_real_sample")
    MetadataCatalog.remove("scannet_real_sample")

DatasetCatalog.register("scannet_real_sample", lambda: load_scannet_flat_5_frames(SCENE_PATH))

# Назначаем классы (ScanNet20)
MetadataCatalog.get("scannet_real_sample").set(
    thing_classes=["wall", "floor", "cabinet", "bed", "chair", "sofa", "table", "door", 
                   "window", "bookshelf", "picture", "counter", "desk", "curtain", 
                   "refrigerator", "showercurtain", "toilet", "sink", "bathtub", "otherfurniture"]
)

print(f"Датасет 'scannet_real_sample' успешно зарегистрирован на основе сцены {SCENE_PATH}!")

In [ ]:
!ls -R /kaggle/working/odin/configs/scannet
!cat /kaggle/working/odin/scripts/scannet/scannet_swin.sh

In [ ]:

!ls -R /kaggle/working/odin/configs/scannet_context

In [ ]:
!pip install "numpy<2" --force-reinstall

In [4]:
import os
import sys
import torch
import numpy as np
import cv2

# 1. ПРОВЕРКА ОКРУЖЕНИЯ И ПУТЕЙ
REPO_PATH = '/kaggle/working/odin'
os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.projects.deeplab import add_deeplab_config
from detectron2.modeling import build_model
from detectron2.data import DatasetCatalog, MetadataCatalog
from odin import add_maskformer2_config, add_maskformer2_video_config

# 2. ФУНКЦИЯ ЗАГРУЗКИ ДАННЫХ
def load_scannet_flat_5_frames(scene_dir, frame_indices=[0, 10, 20, 30, 40]):
    scene_dict = {
        "file_name": os.path.basename(scene_dir), 
        "dataset_name": "scannet_real_sample",
        "images": [], "depths": [], "poses": [], "intrinsics": [],
        "decoder_3d": True 
    }
    
    default_intrinsic = np.array([
        [577.8706, 0.0,      319.5, 0.0],
        [0.0,      577.8706, 239.5, 0.0],
        [0.0,      0.0,      1.0,   0.0],
        [0.0,      0.0,      0.0,   1.0]
    ], dtype=np.float32)
    intrinsic_tensor = torch.from_numpy(default_intrinsic)

    for idx in frame_indices:
        frame_str = f"{idx:05d}"
        
        color_path = os.path.join(scene_dir, f"{frame_str}.jpg")
        img = cv2.imread(color_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        scene_dict["images"].append(torch.from_numpy(img).permute(2, 0, 1).float())
        
        depth_path = os.path.join(scene_dir, f"{frame_str}.png")
        depth_img = cv2.imread(depth_path, cv2.IMREAD_ANYDEPTH) 
        scene_dict["depths"].append(torch.from_numpy(depth_img.astype(np.float32)) / 1000.0)
        
        pose_path = os.path.join(scene_dir, f"{frame_str}.txt")
        with open(pose_path, 'r') as f:
            pose_matrix = np.loadtxt(f)
        if np.isinf(pose_matrix).any():
            pose_matrix = np.eye(4)
        scene_dict["poses"].append(torch.from_numpy(pose_matrix).float())
        scene_dict["intrinsics"].append(intrinsic_tensor)

    return [scene_dict]

# 3. ЖЕСТКАЯ РЕГИСТРАЦИЯ И ПРОВЕРКА МЕТАДАННЫХ
DATASET_NAME = "scannet_real_sample"
SCENE_PATH = "/kaggle/input/scannet-data/scannet/posed_images/scene0000_00"

# Безопасная очистка старых записей
if DATASET_NAME in DatasetCatalog.list():
    DatasetCatalog.remove(DATASET_NAME)
try:
    MetadataCatalog.remove(DATASET_NAME)
except Exception:
    pass

DatasetCatalog.register(DATASET_NAME, lambda: load_scannet_flat_5_frames(SCENE_PATH))
MetadataCatalog.get(DATASET_NAME).set(
    thing_classes=["wall", "floor", "cabinet", "bed", "chair", "sofa", "table", "door", 
                   "window", "bookshelf", "picture", "counter", "desk", "curtain", 
                   "refrigerator", "showercurtain", "toilet", "sink", "bathtub", "otherfurniture"]
)

# КРИТИЧЕСКАЯ ПРОВЕРКА: Если метаданные не записались, код упадет здесь, а не внутри ODIN
assert hasattr(MetadataCatalog.get(DATASET_NAME), 'thing_classes'), "Ошибка: thing_classes не прикрепились к датасету!"
assert len(MetadataCatalog.get(DATASET_NAME).thing_classes) == 20, "Ошибка: неверное количество классов!"

# 4. СКАЧИВАНИЕ ВЕСОВ
os.makedirs("weights", exist_ok=True)
!wget -nc -q https://huggingface.co/katefgroup/odin/resolve/main/scannet_swin_semantic_77.8_64k_2k.pth -O weights/scannet_swin.pth
!wget -nc -q https://huggingface.co/katefgroup/odin/resolve/main/m2f_coco_swin.pkl -O weights/m2f_coco_swin.pkl

# 5. НАСТРОЙКА КОНФИГА
print("Настраиваем конфигурацию...")
cfg = get_cfg()
add_deeplab_config(cfg)
add_maskformer2_config(cfg)
add_maskformer2_video_config(cfg)

cfg.merge_from_file("configs/scannet_context/swin_3d.yaml")

cfg.MODEL.WEIGHTS = "weights/scannet_swin.pth"
cfg.DATASETS.TRAIN = (DATASET_NAME,) # ODIN использует это для получения классов при инициализации
cfg.DATASETS.TEST = (DATASET_NAME,)
cfg.INPUT.SAMPLING_FRAME_NUM = 5
cfg.MODEL.DECODER_3D = True
cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES = 20
cfg.MODEL.CROSS_VIEW_CONTEXTUALIZE = False 

# 6. СБОРКА МОДЕЛИ И ИНФЕРЕНС
print("Инициализируем модель ODIN...")
model = build_model(cfg)
model.eval()  
DetectionCheckpointer(model).resume_or_load(cfg.MODEL.WEIGHTS, resume=False)

print("Грузим 5 реальных кадров и запускаем инференс...")
batched_inputs = load_scannet_flat_5_frames(SCENE_PATH, frame_indices=[0, 10, 20, 30, 40])

with torch.no_grad():
    with torch.cuda.amp.autocast():
        predictions = model(batched_inputs)
        
print("🎉 Инференс завершен успешно!")
print("Доступные ключи в ответе:", predictions[0].keys())

Настраиваем конфигурацию...
Инициализируем модель ODIN...
output_norm GroupNorm(32, 256, eps=1e-05, affine=True)


The checkpoint state_dict contains keys that are not used by the model:
  backbone.layers.1.res_to_trans.0.{bias, weight}
  backbone.layers.1.res_to_trans.1.{bias, weight}
  backbone.layers.1.cross_view_attn.cross_view_attention_layers.0.multihead_attn.{in_proj_bias, in_proj_weight}
  backbone.layers.1.cross_view_attn.cross_view_attention_layers.0.multihead_attn.out_proj.{bias, weight}
  backbone.layers.1.cross_view_attn.cross_view_attention_layers.0.norm.{bias, weight}
  backbone.layers.1.cross_view_attn.cross_view_attention_layers.1.multihead_attn.{in_proj_bias, in_proj_weight}
  backbone.layers.1.cross_view_attn.cross_view_attention_layers.1.multihead_attn.out_proj.{bias, weight}
  backbone.layers.1.cross_view_attn.cross_view_attention_layers.1.norm.{bias, weight}
  backbone.layers.1.cross_view_attn.ffn_layers.0.linear1.{bias, weight}
  backbone.layers.1.cross_view_attn.ffn_layers.0.linear2.{bias, weight}
  backbone.layers.1.cross_view_attn.ffn_layers.0.norm.{bias, weight}
  backbon

Грузим 5 реальных кадров и запускаем инференс...


[ WARN:0@1041.702] global loadsave.cpp:278 findDecoder imread_('/kaggle/input/scannet-data/scannet/posed_images/scene0000_00/00000.jpg'): can't open/read file: check file path/integrity


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


In [ ]:
# /kaggle/input/datasets/tiantiansyrinx1102/scannet-data/scannet/posed_images/scene0000_00

In [7]:
def load_scannet_flat_5_frames(scene_dir, frame_indices=[0, 10, 20, 30, 40]):
    scene_dict = {
        "file_name": os.path.basename(scene_dir), 
        "dataset_name": "scannet_real_sample",
        "images": [], "depths": [], "poses": [], "intrinsics": [],
        "decoder_3d": True 
    }
    
    default_intrinsic = np.array([
        [577.8706, 0.0,      319.5, 0.0],
        [0.0,      577.8706, 239.5, 0.0],
        [0.0,      0.0,      1.0,   0.0],
        [0.0,      0.0,      0.0,   1.0]
    ], dtype=np.float32)
    intrinsic_tensor = torch.from_numpy(default_intrinsic)

    for idx in frame_indices:
        frame_str = f"{idx:05d}"
        
        # 1. Читаем и жмем RGB до 640x480
        color_path = os.path.join(scene_dir, f"{frame_str}.jpg")
        img = cv2.imread(color_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (640, 480), interpolation=cv2.INTER_LINEAR) # <--- ИСПРАВЛЕНИЕ
        scene_dict["images"].append(torch.from_numpy(img).permute(2, 0, 1).float())
        
        # 2. Читаем и жмем Глубину до 640x480 (только NEAREST, чтобы не исказить данные)
        depth_path = os.path.join(scene_dir, f"{frame_str}.png")
        depth_img = cv2.imread(depth_path, cv2.IMREAD_ANYDEPTH) 
        depth_img = cv2.resize(depth_img, (640, 480), interpolation=cv2.INTER_NEAREST) # <--- ИСПРАВЛЕНИЕ
        scene_dict["depths"].append(torch.from_numpy(depth_img.astype(np.float32)) / 1000.0)
        
        pose_path = os.path.join(scene_dir, f"{frame_str}.txt")
        with open(pose_path, 'r') as f:
            pose_matrix = np.loadtxt(f)
        if np.isinf(pose_matrix).any():
            pose_matrix = np.eye(4)
        scene_dict["poses"].append(torch.from_numpy(pose_matrix).float())
        scene_dict["intrinsics"].append(intrinsic_tensor)

    return [scene_dict]

In [9]:
import os
import sys
import torch
import numpy as np
import cv2
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.projects.deeplab import add_deeplab_config
from detectron2.modeling import build_model
from detectron2.data import DatasetCatalog, MetadataCatalog
from odin import add_maskformer2_config, add_maskformer2_video_config

# 1. ПРОВЕРКА ОКРУЖЕНИЯ
REPO_PATH = '/kaggle/working/odin'
os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

CLASSES = ["wall", "floor", "cabinet", "bed", "chair", "sofa", "table", "door", 
           "window", "bookshelf", "picture", "counter", "desk", "curtain", 
           "refrigerator", "showercurtain", "toilet", "sink", "bathtub", "otherfurniture"]

# 2. ФУНКЦИЯ ЗАГРУЗКИ ДАННЫХ + МАТЕМАТИКА 3D-ПРОЕКЦИИ
def load_scannet_flat_5_frames(scene_dir, frame_indices=[0, 10, 20, 30, 40]):
    scene_dict = {
        "file_name": os.path.basename(scene_dir), 
        "dataset_name": "scannet_real_sample",
        "images": [], "depths": [], "poses": [], "intrinsics": [], "valids": [],
        "decoder_3d": True,
        "num_classes": len(CLASSES),
        "all_classes": CLASSES
    }
    
    default_intrinsic = np.array([
        [577.8706, 0.0,      319.5, 0.0],
        [0.0,      577.8706, 239.5, 0.0],
        [0.0,      0.0,      1.0,   0.0],
        [0.0,      0.0,      0.0,   1.0]
    ], dtype=np.float32)
    intrinsic_tensor = torch.from_numpy(default_intrinsic)

    for idx in frame_indices:
        frame_str = f"{idx:05d}"
        
        # RGB
        color_path = os.path.join(scene_dir, f"{frame_str}.jpg")
        img = cv2.imread(color_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (640, 480), interpolation=cv2.INTER_LINEAR)
        scene_dict["images"].append(torch.from_numpy(img).permute(2, 0, 1).float())
        
        # Depth
        depth_path = os.path.join(scene_dir, f"{frame_str}.png")
        depth_img = cv2.imread(depth_path, cv2.IMREAD_ANYDEPTH) 
        depth_img = cv2.resize(depth_img, (640, 480), interpolation=cv2.INTER_NEAREST)
        scene_dict["depths"].append(torch.from_numpy(depth_img.astype(np.float32)) / 1000.0)
        
        # Pose
        pose_path = os.path.join(scene_dir, f"{frame_str}.txt")
        with open(pose_path, 'r') as f:
            pose_matrix = np.loadtxt(f)
        if np.isinf(pose_matrix).any():
            pose_matrix = np.eye(4)
        scene_dict["poses"].append(torch.from_numpy(pose_matrix).float())
        scene_dict["intrinsics"].append(intrinsic_tensor)
        
        # Валидная маска (чтобы модель не ругалась на eval_sparse)
        scene_dict["valids"].append(torch.ones((480, 640), dtype=torch.bool))

    # --- МАТЕМАТИКА 3D: ПРЕВРАЩАЕМ 2D ГЛУБИНУ В 3D ОБЛАКА ТОЧЕК ---
    multi_scale_xyz = []
    original_xyz_frames = []
    scales = [4, 8, 16, 32] # Масштабы, которые требует ODIN
    
    # 1x Масштаб (original_xyz)
    for i in range(len(frame_indices)):
        depth, pose, intr = scene_dict["depths"][i], scene_dict["poses"][i], scene_dict["intrinsics"][i]
        y, x = torch.meshgrid(torch.arange(480), torch.arange(640), indexing='ij')
        X = (x.float() - intr[0,2]) * depth / intr[0,0]
        Y = (y.float() - intr[1,2]) * depth / intr[1,1]
        cam_pts = torch.stack([X, Y, depth], dim=-1).view(-1, 3)
        cam_pts_homo = torch.cat([cam_pts, torch.ones_like(cam_pts[:, :1])], dim=-1)
        world_pts = (pose @ cam_pts_homo.T).T[:, :3].view(480, 640, 3)
        original_xyz_frames.append(world_pts)
    scene_dict["original_xyz"] = torch.stack(original_xyz_frames, dim=0)
    
    # Мульти-масштаб (1/4, 1/8, 1/16, 1/32)
    for scale in scales:
        scale_xyz_frames = []
        Hs, Ws = 480 // scale, 640 // scale
        for i in range(len(frame_indices)):
            depth, pose, intr = scene_dict["depths"][i].unsqueeze(0).unsqueeze(0), scene_dict["poses"][i], scene_dict["intrinsics"][i]
            depth_s = torch.nn.functional.interpolate(depth, size=(Hs, Ws), mode='nearest').squeeze()
            
            y, x = torch.meshgrid(torch.arange(Hs), torch.arange(Ws), indexing='ij')
            y_orig, x_orig = y.float() * scale + (scale - 1) / 2.0, x.float() * scale + (scale - 1) / 2.0
            
            X = (x_orig - intr[0,2]) * depth_s / intr[0,0]
            Y = (y_orig - intr[1,2]) * depth_s / intr[1,1]
            cam_pts = torch.stack([X, Y, depth_s], dim=-1).view(-1, 3)
            cam_pts_homo = torch.cat([cam_pts, torch.ones_like(cam_pts[:, :1])], dim=-1)
            world_pts = (pose @ cam_pts_homo.T).T[:, :3].view(Hs, Ws, 3)
            scale_xyz_frames.append(world_pts)
        multi_scale_xyz.append(torch.stack(scale_xyz_frames, dim=0))
        
    scene_dict["multi_scale_xyz"] = multi_scale_xyz
    # -------------------------------------------------------------

    return [scene_dict]

# 3. РЕГИСТРАЦИЯ ДАТАСЕТА
DATASET_NAME = "scannet_real_sample"
SCENE_PATH = "/kaggle/input/datasets/tiantiansyrinx1102/scannet-data/scannet/posed_images/scene0000_00"

if DATASET_NAME in DatasetCatalog.list():
    DatasetCatalog.remove(DATASET_NAME)
try:
    MetadataCatalog.remove(DATASET_NAME)
except Exception:
    pass

DatasetCatalog.register(DATASET_NAME, lambda: load_scannet_flat_5_frames(SCENE_PATH))
MetadataCatalog.get(DATASET_NAME).set(thing_classes=CLASSES)

# 4. НАСТРОЙКА КОНФИГА
print("Настраиваем конфигурацию...")
cfg = get_cfg()
add_deeplab_config(cfg)
add_maskformer2_config(cfg)
add_maskformer2_video_config(cfg)
cfg.merge_from_file("configs/scannet_context/swin_3d.yaml")

cfg.MODEL.WEIGHTS = "weights/scannet_swin.pth"
cfg.DATASETS.TRAIN = (DATASET_NAME,) 
cfg.DATASETS.TEST = (DATASET_NAME,)
cfg.INPUT.SAMPLING_FRAME_NUM = 5
cfg.MODEL.DECODER_3D = True
cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES = 20
cfg.MODEL.CROSS_VIEW_CONTEXTUALIZE = False 

# ОТКЛЮЧАЕМ зависимость от готовых 3D Mesh файлов!
cfg.USE_GHOST_POINTS = False 
cfg.USE_SEGMENTS = False     

# 5. СБОРКА МОДЕЛИ И ИНФЕРЕНС
print("Инициализируем модель ODIN...")
model = build_model(cfg)
model.eval()  
DetectionCheckpointer(model).resume_or_load(cfg.MODEL.WEIGHTS, resume=False)

print("Грузим 5 реальных кадров и запускаем инференс...")
batched_inputs = load_scannet_flat_5_frames(SCENE_PATH, frame_indices=[0, 10, 20, 30, 40])

with torch.no_grad():
    with torch.cuda.amp.autocast():
        predictions = model(batched_inputs)
        
print("🎉 Инференс завершен успешно!")
print("Доступные ключи в ответе:", predictions[0].keys())

Настраиваем конфигурацию...
Инициализируем модель ODIN...
output_norm GroupNorm(32, 256, eps=1e-05, affine=True)


The checkpoint state_dict contains keys that are not used by the model:
  backbone.layers.1.res_to_trans.0.{bias, weight}
  backbone.layers.1.res_to_trans.1.{bias, weight}
  backbone.layers.1.cross_view_attn.cross_view_attention_layers.0.multihead_attn.{in_proj_bias, in_proj_weight}
  backbone.layers.1.cross_view_attn.cross_view_attention_layers.0.multihead_attn.out_proj.{bias, weight}
  backbone.layers.1.cross_view_attn.cross_view_attention_layers.0.norm.{bias, weight}
  backbone.layers.1.cross_view_attn.cross_view_attention_layers.1.multihead_attn.{in_proj_bias, in_proj_weight}
  backbone.layers.1.cross_view_attn.cross_view_attention_layers.1.multihead_attn.out_proj.{bias, weight}
  backbone.layers.1.cross_view_attn.cross_view_attention_layers.1.norm.{bias, weight}
  backbone.layers.1.cross_view_attn.ffn_layers.0.linear1.{bias, weight}
  backbone.layers.1.cross_view_attn.ffn_layers.0.linear2.{bias, weight}
  backbone.layers.1.cross_view_attn.ffn_layers.0.norm.{bias, weight}
  backbon

Грузим 5 реальных кадров и запускаем инференс...
Number of frames: 5


KeyError: 'instances_all'

In [11]:
import torch
from detectron2.structures import Instances

print("Грузим 5 реальных кадров и готовим батч...")
# Путь и функция загрузки должны быть определены выше
SCENE_PATH = "/kaggle/input/datasets/tiantiansyrinx1102/scannet-data/scannet/posed_images/scene0000_00"
batched_inputs = load_scannet_flat_5_frames(SCENE_PATH, frame_indices=[0, 10, 20, 30, 40])

# Переключаем модель в режим оценки
model.eval()

# Настройка заглушек для каждого кадра в видео-последовательности
for video in batched_inputs:
    # ODIN ожидает, что в батче будет ключ 'instances_all' — список Instances для каждого кадра
    video['instances_all'] = []
    
    # В конфиге для Scannet часто используется множитель для координат (обычно 1000)
    video['multiplier'] = 1000 
    
    for i in range(len(video['images'])):
        # Создаем объект Instances с правильным разрешением
        h, w = video['images'][i].shape[-2:]
        fake_inst = Instances((h, w))
        
        # Добавляем пустые поля, которые требует функция convert_video_instances_to_3d
        # Все они должны быть тензорами Long или Float на нужном устройстве
        fake_inst.gt_classes = torch.tensor([], dtype=torch.long, device=model.device)
        fake_inst.gt_masks = torch.tensor([], dtype=torch.float32, device=model.device)
        fake_inst.instance_ids = torch.tensor([], dtype=torch.long, device=model.device)
        
        video['instances_all'].append(fake_inst)

print("Запускаем инференс... Сейчас модель увидит пустые таргеты и перейдет к предсказаниям. 🪄")
with torch.no_grad():
    with torch.cuda.amp.autocast():
        # Теперь prepare_targets найдет поле 'instance_ids', увидит что оно пустое и пойдет дальше
        predictions = model(batched_inputs)
        
print("🎉 ПОБЕДА! Инференс завершен успешно!")
print("Доступные результаты:", predictions[0].keys())

# Сохраним результат для визуализации
if 'instances_3d' in predictions[0]:
    print(f"Найдено 3D инстансов: {len(predictions[0]['instances_3d']['pred_classes'])}")

Грузим 5 реальных кадров и готовим батч...
Запускаем инференс... Сейчас модель увидит пустые таргеты и перейдет к предсказаниям. 🪄
Number of frames: 5


IndexError: list index out of range

In [13]:
import numpy as np
assert np.__version__.startswith('1.'), "Нужен NumPy 1.x. Выполните: !pip install 'numpy<2' --force-reinstall"

In [14]:
import os
import torch
import numpy as np
import cv2
import sys
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.modeling import build_model
from detectron2.structures import Instances, Boxes
from detectron2.projects.deeplab import add_deeplab_config
from odin import add_maskformer2_config, add_maskformer2_video_config

class ODINPredictor:
    def __init__(self, config_path, weights_path):
        self.cfg = get_cfg()
        add_deeplab_config(self.cfg)
        add_maskformer2_config(self.cfg)
        add_maskformer2_video_config(self.cfg)
        self.cfg.merge_from_file(config_path)
        
        # Настройка параметров инференса
        self.cfg.MODEL.WEIGHTS = weights_path
        self.cfg.MODEL.DECODER_3D = True
        self.cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES = 20
        self.cfg.USE_GHOST_POINTS = False
        self.cfg.USE_SEGMENTS = False
        
        self.model = build_model(self.cfg)
        self.model.eval()
        checkpointer = DetectionCheckpointer(self.model)
        checkpointer.load(weights_path)
        
        self.device = self.model.device
        self.classes = ["wall", "floor", "cabinet", "bed", "chair", "sofa", "table", "door", 
                        "window", "bookshelf", "picture", "counter", "desk", "curtain", 
                        "refrigerator", "showercurtain", "toilet", "sink", "bathtub", "otherfurniture"]

    def _unproject_depth(self, depth, intr, pose):
        """Проецирует карту глубины в 3D облако точек в мировых координатах."""
        h, w = depth.shape
        y, x = torch.meshgrid(torch.arange(h), torch.arange(w), indexing='ij')
        x = x.to(self.device); y = y.to(self.device); depth = depth.to(self.device)
        
        X = (x.float() - intr[0,2]) * depth / intr[0,0]
        Y = (y.float() - intr[1,2]) * depth / intr[1,1]
        
        cam_pts = torch.stack([X, Y, depth], dim=-1).view(-1, 3)
        cam_pts_homo = torch.cat([cam_pts, torch.ones_like(cam_pts[:, :1])], dim=-1)
        world_pts = (pose.to(self.device) @ cam_pts_homo.T).T[:, :3]
        return world_pts.view(h, w, 3)

    def predict(self, frames_data):
        """
        frames_data: список словарей с 'image', 'depth', 'pose', 'intrinsic'
        """
        num_frames = len(frames_data)
        h, w = 480, 640 # Стандарт ScanNet для ODIN
        
        # Подготовка структуры батча
        batch = {
            "file_name": "/tmp/inference/scene/color/0.jpg", # Фиктивный путь для обхода split()[-3]
            "dataset_name": "inference_dataset",
            "images": [], "depths": [], "poses": [], "intrinsics": [],
            "decoder_3d": True, "num_classes": 20, "all_classes": self.classes,
            "instances_all": [], "multiplier": 1000,
            "valids": [torch.ones((h, w), dtype=torch.bool, device=self.device)] * num_frames
        }

        # Ресайз и загрузка тензоров
        for f in frames_data:
            img = cv2.resize(f['image'], (w, h))
            batch["images"].append(torch.from_numpy(img).permute(2, 0, 1).float().to(self.device))
            
            dep = cv2.resize(f['depth'], (w, h), interpolation=cv2.INTER_NEAREST)
            dep_t = torch.from_numpy(dep.astype(np.float32)).to(self.device) / 1000.0
            batch["depths"].append(dep_t)
            
            batch["poses"].append(torch.from_numpy(f['pose']).float().to(self.device))
            batch["intrinsics"].append(torch.from_numpy(f['intrinsic']).float().to(self.device))
            
            # Заглушка для Instances (требуется моделью для внутренней логики)
            inst = Instances((h, w))
            inst.gt_classes = torch.tensor([], dtype=torch.long, device=self.device)
            inst.instance_ids = torch.tensor([], dtype=torch.long, device=self.device)
            batch["instances_all"].append(inst)

        # Расчет multi_scale_xyz (Сердце ODIN)
        original_xyz = []
        for i in range(num_frames):
            xyz = self._unproject_depth(batch["depths"][i], batch["intrinsics"][i], batch["poses"][i])
            original_xyz.append(xyz)
        batch["original_xyz"] = torch.stack(original_xyz)

        scales = [4, 8, 16, 32]
        batch["multi_scale_xyz"] = []
        for s in scales:
            scale_pts = []
            for i in range(num_frames):
                d_s = torch.nn.functional.interpolate(batch["depths"][i][None, None], 
                                                      size=(h//s, w//s), mode='nearest').squeeze()
                xyz_s = self._unproject_depth(d_s, batch["intrinsics"][i], batch["poses"][i])
                scale_pts.append(xyz_s)
            batch["multi_scale_xyz"].append(torch.stack(scale_pts))

        with torch.no_grad():
            with torch.cuda.amp.autocast():
                predictions = self.model([batch])
        
        return predictions[0], batch["original_xyz"]

In [24]:
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.modeling import build_model
from detectron2.structures import Instances
from detectron2.projects.deeplab import add_deeplab_config
from odin import add_maskformer2_config, add_maskformer2_video_config

class ODINPredictor:
    def __init__(self, config_path, weights_path):
        cfg = get_cfg()
        add_deeplab_config(cfg)
        add_maskformer2_config(cfg)
        add_maskformer2_video_config(cfg)
        cfg.merge_from_file(config_path)
        
        # --- КРИТИЧЕСКИЕ НАСТРОЙКИ СОГЛАСОВАННОСТИ ---
        cfg.DATASETS.TRAIN = (DATASET_NAME,)
        cfg.MODEL.WEIGHTS = weights_path
        cfg.MODEL.DECODER_3D = True
        cfg.INPUT.VOXELIZE = True # Обязательно для 3D режима
        cfg.INPUT.VOXEL_SIZE = [0.02, 0.02, 0.02]
        cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES = 20
        cfg.USE_GHOST_POINTS = False
        cfg.USE_SEGMENTS = False
        
        # Соответствие параметров из обучающего скрипта
        cfg.MODEL.MASK_FORMER.TEST.SEMANTIC_ON = True
        cfg.MODEL.MASK_FORMER.TEST.INSTANCE_ON = True
        
        self.model = build_model(cfg)
        self.model.eval()
        DetectionCheckpointer(self.model).load(weights_path)
        
        self.cfg = cfg
        self.device = self.model.device

    def _get_xyz_at_scale(self, depth, intr, pose, scale):
        """Создает 3D облако точек. Правильно учитывает масштабирование интринсиков."""
        h, w = depth.shape
        y, x = torch.meshgrid(torch.arange(h, device=self.device), 
                              torch.arange(w, device=self.device), indexing='ij')
        
        # В ODIN интринсики масштабируются пропорционально разрешению
        fx = intr[0, 0] / scale
        fy = intr[1, 1] / scale
        cx = intr[0, 2] / scale
        cy = intr[1, 2] / scale
        
        X = (x.float() - cx) * depth / fx
        Y = (y.float() - cy) * depth / fy
        
        pts = torch.stack([X, Y, depth], dim=-1).view(-1, 3)
        pts_homo = torch.cat([pts, torch.ones_like(pts[:, :1])], dim=-1)
        world_pts = (pose.to(self.device) @ pts_homo.T).T[:, :3]
        return world_pts.view(h, w, 3)

    @torch.no_grad()
    def predict(self, frames_list):
        h_orig, w_orig = 480, 640
        
        # ODIN Head ожидает ровно 4 масштаба для [res5, res4, res3] и отдельно res2
        # Стандартные коэффициенты уменьшения для этих уровней:
        ratios = [32, 16, 8, 4] 
        
        # Принудительно настраиваем voxel_size в конфиге под 4 уровня, 
        # чтобы избежать AssertionError в multiscsale_voxelize
        self.cfg.INPUT.VOXEL_SIZE = [0.32, 0.16, 0.08, 0.04] 

        batch = {
            "file_name": "inference/scene_00/frame_00.jpg", 
            "dataset_name": DATASET_NAME,
            "images": [], "depths": [], "poses": [], "intrinsics": [],
            "decoder_3d": True, "all_classes": SCANNET_CLASSES, "num_classes": 20,
            "instances_all": [], "multiplier": 1000,
            "multi_scale_xyz": [] 
        }

        # Базовая загрузка (без изменений)
        for f in frames_list:
            img = cv2.resize(f['image'], (w_orig, h_orig))
            batch["images"].append(torch.as_tensor(img.transpose(2,0,1)).float().to(self.device))
            
            depth = cv2.resize(f['depth'], (w_orig, h_orig), interpolation=cv2.INTER_NEAREST)
            depth_m = torch.as_tensor(depth.astype(np.float32)).to(self.device) / 1000.0
            batch["depths"].append(depth_m)
            
            batch["poses"].append(torch.as_tensor(f['pose']).float().to(self.device))
            batch["intrinsics"].append(torch.as_tensor(f['intrinsic']).float().to(self.device))
            
            inst = Instances((h_orig, w_orig))
            inst.gt_classes = torch.tensor([], dtype=torch.long, device=self.device)
            inst.instance_ids = torch.tensor([], dtype=torch.long, device=self.device)
            batch["instances_all"].append(inst)

        # Генерация 4-х масштабов XYZ
        for r in ratios:
            scale_frames = []
            h_s, w_s = h_orig // r, w_orig // r
            for i in range(len(frames_list)):
                d_s = torch.nn.functional.interpolate(
                    batch["depths"][i][None, None], 
                    size=(h_s, w_s), 
                    mode='nearest'
                ).squeeze()
                
                xyz_s = self._get_xyz_at_scale(d_s, batch["intrinsics"][i], batch["poses"][i], r)
                scale_frames.append(xyz_s)
            
            batch["multi_scale_xyz"].append(torch.stack(scale_frames))

        print(f"✅ Подготовлено {len(batch['multi_scale_xyz'])} масштабов XYZ (Ratios: {ratios})")

        with torch.cuda.amp.autocast():
            # Теперь индексы [3] и [:3] в odin_head.py отработают корректно
            output = self.model([batch])[0]
        
        return output, batch["depths"], batch["poses"]

In [22]:
from detectron2.data import MetadataCatalog, DatasetCatalog

# Названия 20 классов ScanNet, которые ожидает предобученная модель
SCANNET_CLASSES = [
    "wall", "floor", "cabinet", "bed", "chair", "sofa", "table", "door", 
    "window", "bookshelf", "picture", "counter", "desk", "curtain", 
    "refrigerator", "showercurtain", "toilet", "sink", "bathtub", "otherfurniture"
]

DATASET_NAME = "scannet_real_sample"

# Очищаем старые записи, если они были
if DATASET_NAME in DatasetCatalog.list():
    DatasetCatalog.remove(DATASET_NAME)
    
# Регистрируем заново с правильными метаданными
DatasetCatalog.register(DATASET_NAME, lambda: []) # Для предиктора нам не нужна функция загрузки здесь
MetadataCatalog.get(DATASET_NAME).set(thing_classes=SCANNET_CLASSES)

print(f"✅ Метаданные для {DATASET_NAME} успешно обновлены!")

✅ Метаданные для scannet_real_sample успешно обновлены!


In [26]:
# 1. Удаляем старые результаты сборки
cd /kaggle/working/odin/libs/pointops2
rm -rf build dist *.egg-info

# 2. Устанавливаем переменные окружения для корректной сборки
export TORCH_CUDA_ARCH_LIST="6.0;7.0;7.5;8.0;8.6" # Поддержка почти всех карт Kaggle (P100, T4)

# 3. Пересобираем и устанавливаем
python setup.py install --user

SyntaxError: invalid syntax (2558664468.py, line 3)

In [25]:
# Инициализация (выполняется один раз)
predictor = ODINPredictor(
    config_path="configs/scannet_context/swin_3d.yaml",
    weights_path="weights/scannet_swin.pth"
)

# Загрузка ваших 5 кадров
SCENE_PATH = "/kaggle/input/datasets/tiantiansyrinx1102/scannet-data/scannet/posed_images/scene0000_00"
frames = []
for i in [0, 10, 20, 30, 40]:
    f_id = f"{i:05d}"
    frames.append({
        'image': cv2.imread(f"{SCENE_PATH}/{f_id}.jpg"),
        'depth': cv2.imread(f"{SCENE_PATH}/{f_id}.png", cv2.IMREAD_ANYDEPTH),
        'pose': np.loadtxt(f"{SCENE_PATH}/{f_id}.txt"),
        'intrinsic': np.array([[577.8, 0, 319.5, 0], [0, 577.8, 239.5, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
    })

# Инференс
output, depths, poses = predictor.predict(frames)

print(f"✅ Успех! Найдено инстансов: {len(output['instances_3d']['pred_classes'])}")



8
8
8
output_norm GroupNorm(32, 256, eps=1e-05, affine=True)


Skip loading parameter 'sem_seg_head.pixel_decoder.cross_view_attn.0.pe_layer.position_embedding_head.0.weight' to the model due to incompatible shapes: (256, 3) in the checkpoint but (256, 3, 1) in the model! You might want to double check if this is expected.
Skip loading parameter 'sem_seg_head.pixel_decoder.cross_view_attn.0.pe_layer.position_embedding_head.3.weight' to the model due to incompatible shapes: (256, 256) in the checkpoint but (256, 256, 1) in the model! You might want to double check if this is expected.
Skip loading parameter 'sem_seg_head.pixel_decoder.cross_view_attn.1.pe_layer.position_embedding_head.0.weight' to the model due to incompatible shapes: (256, 3) in the checkpoint but (256, 3, 1) in the model! You might want to double check if this is expected.
Skip loading parameter 'sem_seg_head.pixel_decoder.cross_view_attn.1.pe_layer.position_embedding_head.3.weight' to the model due to incompatible shapes: (256, 256) in the checkpoint but (256, 256, 1) in the mod

✅ Подготовлено 4 масштабов XYZ (Ratios: [32, 16, 8, 4])
Number of frames: 5


/kaggle/working/odin/libs/pointops2/functions/pointops.py:44: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:83.)
  idx = torch.cuda.IntTensor(m, nsample).zero_()


RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Финальный штрих: Визуализация 3D облака первого кадра с масками
import plotly.graph_objects as go

# Восстанавливаем XYZ первого кадра для отрисовки
h, w = 480, 640
y, x = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
depth0 = depths[0].cpu().numpy()
z = depth0
x_3d = (x - 319.5) * z / 577.8
y_3d = (y - 239.5) * z / 577.8
pts = np.stack([x_3d, y_3d, z], axis=-1).reshape(-1, 3)
colors = output['semantic_3d'].cpu().numpy()

# Отрисовка
fig = go.Figure(data=[go.Scatter3d(
    x=pts[::5, 0], y=pts[::5, 1], z=pts[::5, 2],
    mode='markers',
    marker=dict(size=2, color=colors[::5], colorscale='Viridis')
)])
fig.update_layout(title="ODIN Semantic 3D Reconstruction")
fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Extract the 3D masks and original point cloud logic from the predictions
# Since we used dummy data above, let's visualize the concept of the 3D output

# In a real scenario, ODIN maps the 2D predictions back to the 3D multi_scale_xyz
# Here is how you extract and plot it with Plotly:
def visualize_3d_segmentation(points, colors=None, labels=None):
    """
    points: (N, 3) numpy array of XYZ coordinates
    colors: (N, 3) numpy array of RGB values (0-255)
    labels: (N,) numpy array of class IDs
    """
    # Downsample for visualization if points > 50k to prevent browser crash
    if len(points) > 50000:
        idx = np.random.choice(len(points), 50000, replace=False)
        points = points[idx]
        if labels is not None: labels = labels[idx]
        if colors is not None: colors = colors[idx]

    # Map labels to discrete colors
    unique_labels = np.unique(labels)
    color_map = {lbl: f'rgb({np.random.randint(0,255)}, {np.random.randint(0,255)}, {np.random.randint(0,255)})' for lbl in unique_labels}
    
    marker_colors = [color_map[lbl] for lbl in labels] if labels is not None else colors
    
    fig = go.Figure(data=[go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode='markers',
        marker=dict(
            size=2,
            color=marker_colors,
            opacity=0.8
        )
    )])
    
    fig.update_layout(margin=dict(l=0, r=0, b=0, t=0), scene=dict(aspectmode='data'))
    fig.show()

# Example usage (mocking the extraction from ODIN's output structure):
# real_points = batched_inputs[0]['multi_scale_xyz'][-1].cpu().numpy()
# real_labels = predictions[0]['semantic_3d'].cpu().numpy()
dummy_points = np.random.rand(10000, 3) * 5
dummy_labels = np.random.randint(0, 5, 10000)

visualize_3d_segmentation(dummy_points, labels=dummy_labels)